# SalesMind AI

## Notebook 04 - Retrieval Augmented Generation (RAG)

This notebook builds the company knowledge base using:

- Ollama
- ChromaDB
- LangChain
- Qwen3

In [10]:
import requests

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import OllamaEmbeddings

In [6]:
documents = []

files = [
    "../docs/company_profile.txt",
    "../docs/services.txt",
    "../docs/pricing.txt",
    "../docs/sales_policy.txt"
]

for file in files:
    loader = TextLoader(file, encoding="utf-8")
    documents.extend(loader.load())

print("Documents Loaded:", len(documents))

Documents Loaded: 4


In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Chunks Created:", len(chunks))

Chunks Created: 7


In [8]:
embedding = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("Embedding Model Ready ✅")

Embedding Model Ready ✅


C:\TEMP\ipykernel_30716\1080974703.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embedding = OllamaEmbeddings(


In [9]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="../vector_db"
)

print("Vector Database Created ✅")

Vector Database Created ✅


In [11]:
retriever = vector_db.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever Ready ✅")

Retriever Ready ✅


In [13]:
query = "What AI services does Digital Whopper provide?"

results = retriever.invoke(query)

for i, doc in enumerate(results):
    print("=" * 60)
    print(f"Document {i+1}")
    print(doc.page_content)

Document 1
Digital Whopper is a digital marketing and AI solutions company based in Jaipur, Rajasthan.

The company provides digital transformation services to startups, SMEs, and enterprise clients across India.

Core services include:
Document 2
Digital Whopper Services

1. SEO
Improve Google search rankings using technical SEO, on-page SEO and backlink strategies.

2. Google Ads
Create and optimize Google advertising campaigns.

3. Meta Ads
Run Facebook and Instagram advertising campaigns.

4. Website Development
Design responsive business websites using modern technologies.
Document 3
5. AI Chatbot Development
Develop intelligent chatbots using Large Language Models and Retrieval-Augmented Generation.

6. Social Media Marketing
Manage social media pages and campaigns.

7. Content Marketing
Create blogs, articles and marketing content.

8. Email Marketing
Automate customer engagement using email campaigns.


In [14]:
import requests
import json

def ask_qwen(question, context):

    prompt = f"""
You are an AI Business Consultant for Digital Whopper.

Answer ONLY using the provided company knowledge.

If the answer is not present in the context, say:
"I couldn't find this information in the company knowledge base."

Company Knowledge:
{context}

Question:
{question}
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen3:4b",
            "prompt": prompt,
            "stream": False
        }
    )

    return response.json()["response"]

print("Qwen Connected ✅")

Qwen Connected ✅


In [15]:
question = "What AI services does Digital Whopper provide?"

docs = retriever.invoke(question)

context = "\n\n".join([doc.page_content for doc in docs])

answer = ask_qwen(question, context)

print(answer)

AI Chatbot Development (Develop intelligent chatbots using Large Language Models and Retrieval-Augmented Generation)
